In [ ]:
%%writefile validador_local.py
import os
import time
import sys
from pathlib import Path  # ¡Aquí está la herramienta!

def generar_reporte():
    app_env = os.getenv("APP_ENV", "Desarrollo-Local")
    
    # 1. Definimos la ruta de la carpeta en tu disco usando Path
    # Esto creará una carpeta llamada 'datos_locales' en el mismo directorio de tu notebook
    ruta_carpeta = Path("./datos_locales")
    archivo_reporte = ruta_carpeta / "reporte_salud.txt" # El operador / une rutas de forma limpia
    
    print(f"=== INICIANDO VALIDACIÓN EN ENTORNO: {app_env} ===")
    
    # 2. Replicamos lo que hace Kubernetes: Crear la carpeta física en el disco si no existe
    if not ruta_carpeta.exists():
        print(f"📁 La carpeta '{ruta_carpeta}' no existe. Creándola en el disco duro...")
        ruta_carpeta.mkdir(parents=True, exist_ok=True)
    else:
        print(f"📁 La carpeta '{ruta_carpeta}' ya existe en el disco.")
        
    time.sleep(1)
    print("📊 [TELEMETRÍA] Validando espacio en disco local... OK.")
    
    # 3. Escribimos el archivo de texto en la carpeta física
    try:
        with open(archivo_reporte, "w") as f:
            f.write(f"Reporte de salud para: {app_env}\n")
            f.write("Estado: Carpeta física creada y mapeada con éxito.\n")
        print(f"✅ Copia física escrita con éxito en: {archivo_reporte}")
    except Exception as e:
        print(f"❌ Error al escribir en el disco: {e}")
        sys.exit(1)
        
    print("=== TRABAJO LOCAL FINALIZADO ===")

if __name__ == "__main__":
    generar_reporte()

In [1]:
%%bash
python validador_local.py

=== INICIANDO VALIDACIÓN EN ENTORNO: Desarrollo-Local ===
📁 La carpeta 'datos_locales' no existe. Creándola en el disco duro...
📊 [TELEMETRÍA] Validando espacio en disco local... OK.
✅ Copia física escrita con éxito en: datos_locales/reporte_salud.txt
=== TRABAJO LOCAL FINALIZADO ===


In [4]:
%%bash
# En Linux/Mac/GitBash muestra el contenido de la nueva carpeta
ls -l datos_locales/

total 8
-rw-r--r--  1 admin  staff  93 Jul  6 14:57 reporte_salud.txt


In [7]:
%%bash
kubectl create configmap validador-local-config --from-file=validador_local.py

configmap/validador-local-config created


In [6]:
%%writefile job_local.yaml
apiVersion: batch/v1
kind: Job
metadata:
  name: job-pathlib-batch
  labels:
    component: architect-python-batch
    layer: exercise-3
spec:
  backoffLimit: 1
  template:
    metadata:
      labels:
        component: architect-python-batch
        layer: exercise-3
    spec:
      restartPolicy: Never
      containers:
      - name: python-worker
        image: python:3.10-slim
        # Ejecutamos el script que tiene pathlib
        command: ["python", "/app/validador_local.py"]
        env:
        - name: APP_ENV
          value: "Validacion-Con-Pathlib"
        volumeMounts:
        - name: espacio-codigo
          mountPath: /app
        - name: espacio-datos
          # Montamos el volumen en la ruta exacta que el script va a buscar
          mountPath: /app/datos_locales
      volumes:
      - name: espacio-codigo
        configMap:
          name: validador-local-config
      - name: espacio-datos
        emptyDir: {}

Writing job_local.yaml


In [ ]:
Entonces si cambiamos de decision y queremos enviarla a un contenedor tenemos que hacer el cambio en el punto de montaje a ./data nuevamente?


In [8]:
%%bash
kubectl apply -f job_local.yaml

job.batch/job-pathlib-batch created


In [9]:
%%bash
kubectl logs job/job-pathlib-batch

=== INICIANDO VALIDACIÓN EN ENTORNO: Validacion-Con-Pathlib ===
📁 La carpeta 'datos_locales' no existe. Creándola en el disco duro...
📊 [TELEMETRÍA] Validando espacio en disco local... OK.
✅ Copia física escrita con éxito en: datos_locales/reporte_salud.txt
=== TRABAJO LOCAL FINALIZADO ===
